In [1]:
# Cell 1 — Setup
import os
import glob
import shutil
import subprocess

from natsort import natsorted

# This notebook lives in notebooks/, but INPUT_DIR/OUTPUT_DIR are relative to the
# repo root. Walk up from the current working dir until we find the data folder,
# then chdir there so the notebook works regardless of where the kernel started.
def _find_repo_root(marker=os.path.join("data", "ru")):
    d = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError(
                f"Could not locate repo root containing {marker!r} starting from {os.getcwd()!r}"
            )
        d = parent


os.chdir(_find_repo_root())
print("Working directory:", os.getcwd())

# INPUT_DIR holds one combined .md per sub-document (72.1.337-<N>.ru.md), produced
# by 01_ocr.ipynb. We clean each document in a single pass and write the result
# (same name) into OUTPUT_DIR — flat, no sub-folders.
INPUT_DIR = "data/ru/"
OUTPUT_DIR = "data/ru_cleaned/"
MODEL = "claude-opus-4-8"   # switch to "claude-sonnet-4-6" for cheaper/faster
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fail fast if the Claude CLI is not on PATH.
CLAUDE_BIN = shutil.which("claude")
if CLAUDE_BIN is None:
    raise RuntimeError(
        "The `claude` CLI was not found on PATH. Install Claude Code and make sure "
        "`claude` is runnable from this kernel's environment."
    )
print("claude CLI:", CLAUDE_BIN)

Working directory: /Users/okolobaxa/Documents/Projects/neo4gen
claude CLI: /opt/homebrew/bin/claude


In [2]:
# Cell 2 — Discovery helper (natural sort via natsort)
# One combined file per sub-document: "72.1.337-<N>.ru.md". natsorted() orders by N
# numerically, so 72.1.337-2 precedes 72.1.337-13 (NOT lexicographic).


def sorted_docs(input_dir):
    return natsorted(glob.glob(os.path.join(input_dir, "72.1.337-*.ru.md")))


print("Documents to clean:", [os.path.basename(p) for p in sorted_docs(INPUT_DIR)])

Documents to clean: ['72.1.337-2.ru.md', '72.1.337-8.ru.md', '72.1.337-9.ru.md', '72.1.337-13.ru.md', '72.1.337-14.ru.md', '72.1.337-39.ru.md', '72.1.337-52.ru.md', '72.1.337-53.ru.md', '72.1.337-56.ru.md', '72.1.337-94.ru.md']


In [3]:
# Cell 3 — Cleanup function (one `claude` CLI call per combined document)

SYSTEM_PROMPT = """Ты — текстолог-корректор. Тебе даётся СЫРОЙ РЕЗУЛЬТАТ OCR
одного русского делопроизводственного документа XIX века (дореформенная
орфография). Документ составлен из нескольких подряд идущих страниц, просто
склеенных друг за другом. Текст сильно искажён распознаванием.

Твоя задача — аккуратно восстановить читаемый текст всего документа:
- Исправляй ошибки распознавания символов (например, i/й, В/Б, н/п и подобные).
- Соединяй слова, разорванные переносами, переводами строк и границами страниц.
- Перекомпоновывай обрывки строк в связные предложения и абзацы (исходная разбивка по строкам не важна).
- Удаляй явный «мусор» OCR (одиночные обрывки, случайные символы), если он не несёт текста.
- Если на какой-либо странице есть боковой/маргинальный столбец (краткое содержание
  на полях), отдели его от основного текста и помести под строкой «> [на полях]»
  в конце соответствующего фрагмента.

СТРОГО СОБЛЮДАЙ:
- СОХРАНЯЙ дореформенную орфографию без изменений: ѣ, i, конечный ъ, ѳ, ѵ. НЕ модернизируй написание.
- НЕ придумывай, не переводи, не пересказывай и не дополняй текст. Нечитаемые места оставляй как есть.
- Выводи ТОЛЬКО исправленный текст документа. Без вступлений, без комментариев, без ограждающих ``` блоков."""


def _strip_fences(text):
    """Defensively remove a leading/trailing ``` code fence if the model wrapped output."""
    s = text.strip()
    if s.startswith("```"):
        lines = s.split("\n")
        lines = lines[1:]  # drop opening fence line (``` or ```lang)
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]  # drop closing fence line
        s = "\n".join(lines)
    return s


def clean_document(ocr_text):
    proc = subprocess.run(
        [CLAUDE_BIN, "-p", f"ДОКУМЕНТ ДЛЯ ОБРАБОТКИ:\n{ocr_text}", "--model", MODEL,
         "--append-system-prompt", SYSTEM_PROMPT],
        capture_output=True, text=True, timeout=1800,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"claude failed (rc={proc.returncode}): {proc.stderr.strip()}")
    return _strip_fences(proc.stdout).strip()

In [4]:
# Cell 4 — Cleanup loop
# --- Config ---
LIMIT = None        # process only the first N documents; set None to process all
REPROCESS = False   # True = re-clean and overwrite even if a cleaned file already exists

from IPython.display import display

files = sorted_docs(INPUT_DIR)
if LIMIT is not None:
    files = files[:LIMIT]

total = len(files)
display(f"Cleaning {total} document(s)  (LIMIT={LIMIT}, REPROCESS={REPROCESS}, MODEL={MODEL})")

for i, input_path in enumerate(files, start=1):
    name = os.path.basename(input_path)
    out_path = os.path.join(OUTPUT_DIR, name)

    if os.path.exists(out_path) and not REPROCESS:
        display(f"[{i}/{total}] skip  {name} (already cleaned)")
        continue

    display(f"[{i}/{total}] clean {name} ...")
    # No try/except by design: an error halts the loop so the failing document can be
    # inspected. Re-running later skips completed files and resumes here.
    with open(input_path, encoding="utf-8") as f:
        ocr_text = f.read()
    cleaned = clean_document(ocr_text)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(cleaned)
    display(f"[{i}/{total}] done  {name}")

display("Finished.")

'Cleaning 10 document(s)  (LIMIT=None, REPROCESS=False, MODEL=claude-opus-4-8)'

'[1/10] clean 72.1.337-2.ru.md ...'

'[1/10] done  72.1.337-2.ru.md'

'[2/10] clean 72.1.337-8.ru.md ...'

'[2/10] done  72.1.337-8.ru.md'

...

'[9/10] clean 72.1.337-56.ru.md ...'

'[9/10] done  72.1.337-56.ru.md'

'[10/10] clean 72.1.337-94.ru.md ...'

'[10/10] done  72.1.337-94.ru.md'

'Finished.'